# Phase 16 — Pipeline Validation (SQL + NoSQL combined)

Runs the new `src/pipeline_sql.py` / `src/pipeline_nosql.py` library code (Phase 16)
end-to-end on whatever GPU this session gets — **works unmodified on T4 or A100**.
`GeneratorInfer` (`src/generator/infer.py`) loads the 7B model in int8 (bitsandbytes)
on any GPU under 20GiB so it fits entirely in VRAM on a T4, and full bf16 at 20GiB and
above where it fits without quantization — see the comments in that file from the
Phase 14/15 OOM and CPU-offload fixes. (The cut is at 20GiB, not 24: nominal-24GB cards
report ~22–23.6GiB usable and belong on the bf16 path.) No code changes needed to switch
GPU type; **T4 will be a bit slower** (int8 vs bf16) than A100, not offloaded to CPU and hung.

Two things this notebook checks that `Phase15_POSG_Combined.ipynb` didn't:
1. **Direct calls into `src.pipeline_sql.run_pipeline()` / `src.pipeline_nosql.run_pipeline()`**
   (not just the `scripts/run_posg_*.py` CLI) — confirms the new Phase 16 library
   functions work standalone, since that's what Phase 17's LangGraph router will call.
2. **Which GPU this session actually got**, printed up front, so you know whether to
   expect A100-speed (bf16) or T4-speed (int8) generation before waiting on anything.

See `docs/phase15_posg_findings.md` for the full Phase 15 methodology and EX results
this notebook's smoke-test cells reproduce.

## 0. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        total_gb = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        print(f"GPU {i}: {name}  ({total_gb:.1f} GB)")
        # Mirrors the 20GiB cut in src/generator/infer.py -- keep the two in sync.
        if total_gb >= 20:
            print("  -> >=20GiB (A100/L4-class; A100 ~15 CU/hr). GeneratorInfer loads full "
                  "bf16 -- fastest path, but this notebook is inference-only (single "
                  "questions + n=30 smoke tests) and doesn't need A100 headroom. T4 covers "
                  "it at ~1.9 CU/hr (~8x cheaper); consider Runtime > Change runtime type > "
                  "T4 GPU unless you specifically want the speed.")
        else:
            print("  -> <20GiB (T4-class, ~1.9 CU/hr): right choice for this notebook. "
                  "GeneratorInfer loads the 7B model in int8 (bitsandbytes) so it fits "
                  "entirely in VRAM -- no CPU offload, no hang. See src/generator/infer.py.")
else:
    print("No GPU detected -- runtime will fall back to CPU, which is impractically slow "
          "for a 7B model doing 5-candidate generation. Set Runtime > Change runtime type > GPU.")

## 1. Clone repo + install dependencies

In [2]:
!git clone https://github.com/kethansplunk/Codegen.git
%cd Codegen
!pip install -q torch transformers peft sqlparse pyyaml FlagEmbedding chromadb openai python-dotenv pymongo bitsandbytes

Cloning into 'Codegen'...
remote: Enumerating objects: 1128, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 1128 (delta 109), reused 122 (delta 50), pack-reused 931 (from 1)
Receiving objects: 100% (1128/1128), 17.78 MiB | 27.33 MiB/s, done.
Resolving deltas: 100% (828/828), done.
/content/Codegen
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 132.6 MB/s eta 

## 2. Mount Drive and load checkpoints (both tracks)

Loads SAR + Generator checkpoints for **both** SQL and NoSQL from Drive. If your Drive layout differs from `codegen/checkpoints/{sar,generator}_{sql,nosql}`, adjust `DRIVE` below.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/codegen'
os.makedirs('models', exist_ok=True)

for name in ['sar_sql', 'generator_sql', 'sar_nosql', 'generator_nosql']:
    dst = f'models/{name}'
    if not os.path.exists(dst):
        os.symlink(f'{DRIVE}/checkpoints/{name}', dst)

!ls -la models/sar_sql models/generator_sql models/sar_nosql models/generator_nosql

Mounted at /content/drive
lrwxrwxrwx 1 root root 58 Jul 19 07:48 models/generator_nosql -> /content/drive/MyDrive/codegen/checkpoints/generator_nosql
lrwxrwxrwx 1 root root 56 Jul 19 07:48 models/generator_sql -> /content/drive/MyDrive/codegen/checkpoints/generator_sql
lrwxrwxrwx 1 root root 52 Jul 19 07:48 models/sar_nosql -> /content/drive/MyDrive/codegen/checkpoints/sar_nosql
lrwxrwxrwx 1 root root 50 Jul 19 07:48 models/sar_sql -> /content/drive/MyDrive/codegen/checkpoints/sar_sql


In [4]:
# Safety net: force sar.backend to memory. ChromaDB's PersistentClient can't open
# an index over a Google Drive FUSE mount, so this avoids that failure mode entirely.
text = open('configs/config.yaml').read()
text = text.replace('backend: chroma', 'backend: memory')
open('configs/config.yaml', 'w').write(text)
!grep -A1 "^sar:" configs/config.yaml | head -3

sar:
  # "memory" → SARRetriever: re-encodes corpus at startup (~30 sec). No ChromaDB needed.


## 3. Spider SQLite databases

Needed for SQL EX scoring, and as the source data for the MongoDB conversion below. Uploaded once as a zip to Drive (see `docs/phase15_posg_findings.md` for how it was built, from a sibling local project).

In [5]:
!cp /content/drive/MyDrive/codegen/checkpoints/spider_database.zip /content/Codegen/
!unzip -q /content/Codegen/spider_database.zip -d /content/Codegen/Data/Spider/
!ls /content/Codegen/Data/Spider/database | wc -l

166


## 4. MongoDB setup (needed for NoSQL EX)

**Important**: `Data/mongodb/*.json` schema-cache files are git-tracked from an earlier run on a different machine. `convert_all()` treats their existence as "already converted" and will silently skip real data insertion into this fresh session's empty `mongod` if we don't clear them first (see `docs/phase15_posg_findings.md`).

In [13]:
# Install and start mongod
!apt-get install -y mongodb >/dev/null 2>&1 || (curl -fsSL https://pgp.mongodb.com/server-7.0.asc | sudo gpg -o /usr/share/keyrings/mongodb-server-7.0.gpg --dearmor && echo "deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list && apt-get update -qq && apt-get install -y mongodb-org)
!mkdir -p /data/db
import subprocess, time
subprocess.Popen(["mongod", "--dbpath", "/data/db", "--bind_ip", "127.0.0.1"])
time.sleep(5)
!mongosh --eval "db.version()"

deb [signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg arch=amd64] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
The following NEW packages will be installed:
  mongodb-database-tools mongodb-mongosh mongodb-org mongodb-org-database
  mongodb-org-database-tools-extra mongodb-org-mongos mongodb-org-server
  mongodb-org-shell mongodb-org-tools
0 upgraded, 9 newly installed, 0 to remove and 145 not upgraded.
Need to get 189 MB of archives.
After this operation,

In [14]:
import shutil
shutil.rmtree('Data/mongodb', ignore_errors=True)   # force a real reconversion, see note above

from src.mongodb_converter import convert_all
convert_all(
    db_root="Data/Spider/database",
    fk_graph_dir="Data/fk_graphs",
    schema_cache_dir="Data/mongodb",
)

[1/166] academic — 15 collections, 42 fields
[2/166] activity_1 — 5 collections, 22 fields
[3/166] aircraft — 5 collections, 28 fields
[4/166] allergy_1 — 3 collections, 12 fields
[5/166] apartment_rentals — 6 collections, 31 fields
[6/166] architecture — 3 collections, 17 fields
[7/166] assets_maintenance — 14 collections, 64 fields
[8/166] baseball_1 — 26 collections, 352 fields
[9/166] battle_death — 3 collections, 18 fields
[10/166] behavior_monitoring — 11 collections, 64 fields
[11/166] bike_1 — 4 collections, 46 fields
[12/166] body_builder — 2 collections, 11 fields
[13/166] book_2 — 2 collections, 9 fields
[14/166] browser_web — 3 collections, 11 fields
[15/166] candidate_poll — 2 collections, 14 fields
[16/166] car_1 — 6 collections, 23 fields
[17/166] chinook_1 — 11 collections, 64 fields
[18/166] cinema — 3 collections, 17 fields
[19/166] city_record — 4 collections, 27 fields
[20/166] climbing — 2 collections, 12 fields
[21/166] club_1 — 3 collections, 15 fields
[22/166] c

In [15]:
# Verify real data landed (not just schema cache) before trusting any EX result
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
print(len(client.list_database_names()), client.list_database_names()[:10])
print("formula_1.drivers count:", client['formula_1']['drivers'].count_documents({}))   # must be > 0

169 ['academic', 'activity_1', 'admin', 'aircraft', 'allergy_1', 'apartment_rentals', 'architecture', 'assets_maintenance', 'baseball_1', 'battle_death']
formula_1.drivers count: 842


## 5. DeepSeek API key

Needed by the `SchemaLinker` call inside `run_pipeline()` in sections 6 and 8 below, so it
has to run *before* them for a clean top-to-bottom execution. The smoke tests in sections 7
and 9 reuse pre-computed `key_fields` and never call DeepSeek.

Paste your real key in place of the placeholder, then re-run this cell — it will not
overwrite a key that's already there. **Never commit this cell with a real key filled in.**

In [ ]:
from pathlib import Path

PLACEHOLDER = 'your_actual_key_here'
env = Path('.env')
existing = env.read_text() if env.exists() else ''

# Re-running this cell must not clobber a key you've already pasted in.
if 'DEEPSEEK_API_KEY=' in existing and PLACEHOLDER not in existing:
    print('.env already contains a DEEPSEEK_API_KEY -- left untouched.')
else:
    env.write_text(f'DEEPSEEK_API_KEY={PLACEHOLDER}\n')
    print('Wrote .env with a placeholder. Edit it (or this cell) with your real key '
          'before running sections 6 and 8, or SchemaLinker will fail there.')

## 6. SQL track — direct `run_pipeline()` call (Phase 16 library test)

Calls `src.pipeline_sql.run_pipeline()` directly in-process (not via the CLI script) — this
is exactly what Phase 17's LangGraph router will do, so a clean run here is the real Phase 16
validation. Loads `configs/config.yaml`, builds the schema for `concert_singer` internally,
runs SchemaLinker -> SAR -> Generator (5 candidates) -> POSG, and returns the selected SQL.

Each `run_pipeline()` call builds SchemaLinker + SAR + the 7B generator from scratch, which is fine for a one-shot check like this but costs tens of seconds and several GB of VRAM per call. For repeated questions (Phase 17's router), build those once and pass them in via the `linker=` / `sar=` / `generator=` parameters.

In [12]:
import yaml
from src.pipeline_sql import run_pipeline

config = yaml.safe_load(open("configs/config.yaml"))

result = run_pipeline(
    question="How many singers are there?",
    db_name="concert_singer",
    config=config,
    strategy="balanced",
)

print("candidates:")
for c in result["candidates"]:
    print(" -", c)
print("\ngreedy:  ", result["greedy"])
print("selected:", result["selected"])
print("posg_diverged:", result["posg_diverged"])

Running on: cuda


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

candidates:
 - SELECT count(*) FROM singer
 - SELECT count(*) FROM singer
 - SELECT count(*) FROM singer
 - SELECT count(*) FROM singer
 - SELECT count(*) FROM singer

greedy:   SELECT count(*) FROM singer
selected: SELECT count(*) FROM singer
posg_diverged: False


## 7. SQL track — full dev-set smoke test (all difficulty, Phase 15 baseline)

`sql_dev_eval_full.json` (1034 Spider dev questions, real SchemaLinker `key_fields`) was rebuilt locally on Mac after the Colab runtime that originally built it disconnected. Upload it to Drive from your Mac first, then pull it in here. This run (no `--hard` filter) is the one comparable to the plan's >82% EX target -- reproduces the Phase 15A numbers using the (now-refactored) CLI, doubling as a regression check that the Phase 16 extraction didn't change behavior.

In [16]:
%%time
# Timed so you can extrapolate to the full 1034-question dev set before
# committing compute-unit budget to it: (this cell's wall time / 30) * 1034.
!cp /content/drive/MyDrive/codegen/sql_dev_eval_full.json Data/cot_data/sql_dev_eval_full.json
!python -m scripts.run_posg_sql --smoke_test --n 30 --data Data/cot_data/sql_dev_eval_full.json

cp: cannot stat '/content/drive/MyDrive/codegen/sql_dev_eval_full.json': No such file or directory
Running on: cuda
Data source: Data/cot_data/sql_dev_eval_full.json
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2802.22it/s]
Loaded corpus: 7000 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 28/28 [00:00<00:00, 90.00it/s]
Inference Embeddings: 100% 28/28 [00:00<00:00, 33.25it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.08it/s] 

[1/30] What are the name, independence year, and surface area of the country with the smallest population?
  gold:     SELECT Name ,  SurfaceArea ,  IndepYear FROM country ORDER BY Population LIMIT 1
  greedy:   SELECT Name ,  IndepYear ,  SurfaceArea FROM country ORDER BY Population ASC LIMIT 1
  selected: SELECT Name ,  IndepYear ,  SurfaceArea FROM country ORDER BY Population ASC LIMIT 1  [same as greedy]
  pareto_front_size=5  

## 8. NoSQL track — direct `run_pipeline()` call (Phase 16 library test)

Same idea as section 6, calling `src.pipeline_nosql.run_pipeline()` directly. Requires the live `mongod` + loaded databases from section 4.

In [18]:
from src.pipeline_nosql import run_pipeline as run_pipeline_nosql

result_nosql = run_pipeline_nosql(
    question="How many singers are there?",
    db_name="concert_singer",
    config=config,
    strategy="balanced",
)

import json
print("candidates:")
for c in result_nosql["candidates"]:
    print(" -", json.dumps(c, ensure_ascii=False))
print("\ngreedy:  ", json.dumps(result_nosql["greedy"], ensure_ascii=False))
print("selected:", json.dumps(result_nosql["selected"], ensure_ascii=False))
print("posg_diverged:", result_nosql["posg_diverged"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...


Inference Embeddings: 100%|██████████| 23/23 [00:00<00:00, 34.58it/s]


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

candidates:
 - {"collection": "singer", "pipeline": [{"$count": "count"}]}
 - {"collection": "singer", "pipeline": [{"$count": "count"}]}
 - {"collection": "singer", "pipeline": [{"$count": "count"}]}
 - {"collection": "singer", "pipeline": [{"$count": "count"}]}
 - {"collection": "singer", "pipeline": [{"$count": "count"}]}

greedy:   {"collection": "singer", "pipeline": [{"$count": "count"}]}
selected: {"collection": "singer", "pipeline": [{"$count": "count"}]}
posg_diverged: False


## 9. NoSQL track — train-split smoke test (all difficulty, Phase 15 baseline)

Already validated: EX 76.7% (POSG) vs 73.3% (greedy) on this exact command (Phase 15B). Re-run here to confirm reproducibility after the Phase 16 refactor.

In [19]:
%%time
!python -m scripts.run_posg_nosql --smoke_test --n 30

Running on: cuda
Data source: Data/cot_data/nosql_cot_train.json
Mongo URI: mongodb://localhost:27017  (make sure mongod is running and target DBs are loaded)
Loading SAR retriever ...
Loading weights: 100% 391/391 [00:00<00:00, 2916.19it/s]
Loaded corpus: 5697 entries
Pre-computing corpus embeddings ...
pre tokenize: 100% 23/23 [00:00<00:00, 197.79it/s]
Inference Embeddings: 100% 23/23 [00:00<00:00, 31.99it/s]
Loading Generator ...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:03<00:00, 88.65it/s] 

[1/30] show the titles, and authors or editors for all books made after the year 1989.
  gold:     {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Editor", "_id": 0}}]}
  greedy:   {"collection": "book_club", "pipeline": [{"$match": {"Year": {"$gt": 1989}}}, {"$project": {"book_title": "$Book_Title", "author_or_editor": "$Author_or_Ed

## 10. Free the GPU when you're done

Units keep burning as long as the runtime stays connected, even idle. Run
this once you've read the output above and don't need the session anymore —
it disconnects and releases the GPU. (Not run automatically: you may still
want to inspect variables or re-run a cell first.)

In [21]:
from google.colab import runtime
runtime.unassign()